In [3]:
import torch
# from .transform import *
from amr.dataloaders.transform import *
import numpy as np

__all__ = ['DataPreprocess']


class DataPreprocess(object):
    '''
    根据指定的处理模式对信号数据进行预处理，是调制识别任务中的关键数据转换组件。
    '''

    def __init__(self, Xmode):
        super(DataPreprocess, self).__init__()

        # Xmode: 包含预处理类型和选项的字典
        self.Xmode = Xmode
        # 提取预处理类型
        self.Xtype = self.Xmode["type"]

    def datapreprocess(self, X):  # 根据Xmode定制批数据

        # 复制输入数据，避免修改原始数据
        NX = X.copy()

        # 应用IQ归一化（如果配置中启用）
        if ("IQ_norm" in self.Xmode["options"]) and self.Xmode["options"]["IQ_norm"]:
            X = normalize_IQ(X)

        # 应用零掩码（如果配置中启用）
        if ("zero_mask" in self.Xmode["options"]) and self.Xmode["options"]["zero_mask"]:
            X = zero_mask(X)

        # 根据不同的预处理类型应用特定转换

        # IQ_framed: 提取IQ帧数据
        if self.Xmode["type"] == "IQ_framed":
            X = get_iq_framed(NX)

        # AP: 转换为幅度-相位表示
        if self.Xmode["type"] == "AP":
            X = get_amp_phase(NX)

        # APF: 转换为幅度-相位-频率表示
        if self.Xmode["type"] == "APF":
            X = get_apf1(NX)

        # APF_ours: 自定义的幅度-相位-频率表示
        if self.Xmode["type"] == "APF_ours":
            X = get_apf2(NX)

        return X

    def __call__(self, X):
        # 实现可调用对象协议
        # 允许直接调用类实例进行数据预处理
        return self.datapreprocess(X)


if __name__ == "__main__":
    # 示例配置：使用自定义的"star"类型预处理
    # 启用IQ归一化，禁用零掩码
    Xmode = {"type":"star","options":{"IQ_norm":True, "zero_mask":False, "img_size":[224,224]}}
    # 创建预处理实例
    pre = DataPreprocess(Xmode)
    # 生成随机测试数据（3个样本，每个样本2通道，长度20）
    x = np.random.randn(3, 2, 20)
    # 应用预处理
    x_pre = pre(x)